策略方向筛选复算

本笔记本读取35个已冻结研究账户，不改变信号或重跑选优。历史区间2021-09-01—2025-08-06；不是未触碰盲测。资金20万元，佣金万五，最低5元。独立源价格账务复算已保存。

In [1]:
from pathlib import Path
import json, hashlib
import pandas as pd
root = Path('/Users/guo/Documents/stocks/quant-research')
run = root / 'artifacts/strategy-screen-20261013-v1'
audit = json.loads((run / 'independent-audit.json').read_text())
verified = 0
for path, expected in audit['source_hashes'].items():
    if '/results/' in path or path.endswith(('protocol.json', 'implementation-freeze.json')):
        assert hashlib.sha256((root / path).read_bytes()).hexdigest() == expected, path
        verified += 1
print({'verified_bindings': verified, 'independently_audited_accounts': audit['accounts'], 'sealed_price_rows_read': audit['sealed_price_rows_read']})


{'verified_bindings': 217, 'independently_audited_accounts': 35, 'sealed_price_rows_read': 0}


In [2]:
rows = pd.read_csv(run / 'results/comparison.csv')
base = rows.loc[rows.scenario.eq('base'), ['strategy','net_return','cagr','max_drawdown','profit_cny','commission','minimum_fee_fills']]
print(base.to_string(index=False))
for item in rows.itertuples():
    nav = pd.read_parquet(run / f'results/{item.scenario}/{item.strategy}/nav.parquet')
    values = pd.concat([pd.Series([200000.]), nav.equity], ignore_index=True)
    net = values.iloc[-1]/200000.-1
    mdd = -(values/values.cummax()-1).min()
    assert abs(net-item.net_return)<1e-12
    assert abs(mdd-item.max_drawdown)<1e-12
print('All 35 net returns and daily maximum drawdowns independently recomputed.')


  strategy  net_return      cagr  max_drawdown     profit_cny  commission  minimum_fee_fills
momentum20   -0.937950 -0.521264      0.952538 -187589.902101 3188.656012                306
  trend120   -0.902955 -0.461031      0.915193 -180591.044642 2388.827727                282
  lowvol60    0.563471  0.125721      0.129098  112694.211327 7714.948652                  0
 reversal5    0.268431  0.065036      0.308235   53686.267531 7689.156842                  0
liquidity6   -0.668748 -0.253810      0.703950 -133749.571382 2308.132939                 82
All 35 net returns and daily maximum drawdowns independently recomputed.


In [3]:
phase = rows[rows.scenario.isin(['base','phase5','phase10','phase15'])]
print(phase.pivot(index='strategy',columns='scenario',values='net_return').to_string())
print(json.dumps(json.loads((run/'triage.json').read_text())['decisions'],ensure_ascii=False,indent=2))


scenario        base   phase10   phase15    phase5
strategy                                          
liquidity6 -0.668748 -0.733629 -0.789984 -0.705311
lowvol60    0.563471  0.129432  0.092028  0.327147
momentum20 -0.937950 -0.947194 -0.911764 -0.930195
reversal5   0.268431 -0.336284 -0.268037 -0.123900
trend120   -0.902955 -0.874549 -0.850986 -0.878635
[
  {
    "strategy": "lowvol60",
    "decision": "continue_research",
    "description": "保留，唯一优先验证方向",
    "continue_gate_passed": true,
    "positive_phase_count": 4,
    "phase_count": 4,
    "positive_calendar_buckets": 4,
    "calendar_buckets": 5,
    "partial_calendar_buckets": [
      "2021",
      "2025"
    ],
    "worst_phase_drawdown": 0.2223705366554591,
    "formal_profit_qualified": false
  },
  {
    "strategy": "reversal5",
    "decision": "deprioritize",
    "description": "暂停当前版本，不投入主策略研发",
    "continue_gate_passed": false,
    "positive_phase_count": 1,
    "phase_count": 4,
    "positive_calendar_buckets": 3,
   

口径和边界见同名研究报告。原快照为重建研究数据；推算涨跌停、分红到账和送转股份锁定等假设限制实盘适用性。低波动通过的是继续研究门槛，未建立稳定收益或实盘合格结论。